# Track A, steps A1 to A3: image pool, pairs, baselines
Runs on a free Colab CPU or GPU runtime. Run top to bottom. Status: the pair/baseline code is tested on synthetic images; the **download cells (Amazon, Stanford Online Products) were not run against the real servers**, so check each printed count before moving on.

In [ ]:
# 0) setup: edit GITHUB_USER, then run
GITHUB_USER = "YOUR_GITHUB_USERNAME"
import os, sys
if not os.path.exists("/content/returnlens"):
    !git clone https://github.com/{GITHUB_USER}/returnlens.git /content/returnlens   # private repo: use a token URL, or just upload the src/ folder
%cd /content/returnlens
!git checkout track-a
!pip -q install opencv-python-headless imagehash scikit-learn pandas lightgbm
sys.path.insert(0, "/content/returnlens/src")
import random, numpy as np, pandas as pd
random.seed(0); np.random.seed(0)
from google.colab import drive; drive.mount("/content/drive")      # only for saving results; images stay on /content
OUT = "/content/drive/MyDrive/returnlens_results"; os.makedirs(OUT, exist_ok=True)

## A1a. Amazon review photos (natural hard negatives)
Products with photos from at least 2 different customers. Start small (`max_products=100`) to test, then raise to ~400. Dead image URLs are skipped.

In [ ]:
import pool
amz = pool.amazon_pool("Electronics", "/content/img_amazon", max_lines=300_000, max_products=100)
print(len(amz), "images,", len({p["product_id"] for p in amz}), "products")
assert len(amz) > 50, "too few images: try another category (Cell_Phones_and_Accessories, Home_and_Kitchen) or a larger max_lines"

## A1b. Stanford Online Products
Download the zip from the [SOP homepage](https://ai.stanford.edu/~sijia/Stanford-Online-Products/) (about 2.9 GB; use the link on that page). Easiest: download on your laptop, upload to Google Drive, unzip here. Files are named `<productid>_<n>.JPG`.

In [ ]:
SOP_ZIP = "/content/drive/MyDrive/Stanford_Online_Products.zip"       # edit to where you put it
if os.path.exists(SOP_ZIP):
    !unzip -q -o "{SOP_ZIP}" -d /content/sop
    sop = pool.folder_pool("/content/sop", max_products=150)
else:
    sop = []; print("SOP zip not found: continuing with Amazon only")
print(len(sop), "SOP images")

In [ ]:
POOL = amz + sop
print(len(POOL), "total images,", len({p["product_id"] for p in POOL}), "products")

## A1c. Pair table
Positives: 3 edited copies of each image. Hard negative: another photo of the **same product**. Easy negative: a random image. Takes a few minutes on CPU.

In [ ]:
from build_pairs import build_pairs
df = build_pairs(POOL, n_variants=3)
print(df.kind.value_counts())
df.to_csv(f"{OUT}/pairs.csv", index=False)

## A1d. Split by product (never by row)

In [ ]:
tr, va, te = pool.split_by_group(df)
for n, d in (("train", tr), ("val", va), ("test", te)):
    print(n, len(d), "pairs,", d.group.nunique(), "products,", d.kind.value_counts().to_dict())

## A2. Eyeball check
Look at matched pairs before trusting any number. Matches should sit on the same object.

In [ ]:
import cv2, matplotlib.pyplot as plt
from evidence import transform
def show_match(path):
    a = cv2.imread(path); b, ops = transform(a)
    g = lambda x: cv2.cvtColor(x, cv2.COLOR_BGR2GRAY)
    orb = cv2.ORB_create(1000); ka, da = orb.detectAndCompute(g(a), None); kb, db = orb.detectAndCompute(g(b), None)
    if da is None or db is None or len(da) < 2 or len(db) < 2: return None
    m = [x for x, y in cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(da, db, k=2) if x.distance < 0.75 * y.distance]
    return cv2.drawMatches(a, ka, b, kb, m[:60], None, flags=2), "+".join(ops), len(m)
fig = plt.figure(figsize=(14, 10))
for i, p in enumerate(random.sample(POOL, min(4, len(POOL)))):
    r = show_match(p["path"])
    if r: ax = fig.add_subplot(2, 2, i + 1); ax.imshow(cv2.cvtColor(r[0], cv2.COLOR_BGR2RGB)); ax.set_title(f"edit: {r[1]}  good matches: {r[2]}"); ax.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/matches.png", dpi=110); plt.show()

## A3. Baselines table (your minimum deliverable)
Thresholds are picked on **val only** so that false positives on hard negatives stay under 1%, then reported on test. Do not retune after seeing test.

In [ ]:
from baselines import table
res = table(va, te)
display(res)
res.to_csv(f"{OUT}/baselines_table.csv", index=False)

**Read this table honestly.** If pHash or ORB reach ~100% recall with ~0% hard-negative false positives, the edits are probably too easy or the negatives too different: check the `kind` counts and look at some hard-negative pairs. Per-transformation recall is next.

In [ ]:
from baselines import pick_threshold
def recall_by_op(test, col, thr):
    pos = test[test.kind == "same_source"].copy()
    pos["hit"] = pos[col] >= thr
    rows = []
    for op in ["resize", "rotate", "crop", "jpeg", "bright", "blur", "border"]:
        s = pos[pos.ops.str.contains(op)]
        if len(s): rows.append(dict(op=op, n=len(s), recall=round(s.hit.mean(), 3)))
    return pd.DataFrame(rows)
for col in ("phash_sim", "orb_inliers"):
    print(col); display(recall_by_op(te, col, pick_threshold(va, col)))